# tpu-as-gpu — Colab TPU verification

Runtime > Change runtime type > TPU, then Runtime > Run all.

This is the exact script run interactively via `colab-mcp` on a Colab TPU
v5e-1 runtime on 2026-10-08 — results in `results/colab_run_2026-10-08.md`.
Mirrors `examples/kaggle/verify_kernel.py` with two Colab-specific
differences: `tensorflow` isn't preinstalled here (Kaggle's TPU VM image
ships a TPU-enabled build, Colab's base image doesn't — see
`research/prior_art.md` §6), and the TPU resolver needs `tpu="local"`
since this is a TPU-VM-style runtime, not the classic Colab TPU-node
address pattern.

In [ ]:
# tensorflow isn't preinstalled on Colab's TPU runtime. Installing it gets
# you CPU-only kernels though (see research/prior_art.md §6) - this cell
# still runs it so the gap is visible in the output rather than silently
# skipped.
!pip install -q tensorflow

import os
print({k: v for k, v in os.environ.items() if "TPU" in k})

In [ ]:
# --- inlined tpu_as_gpu (mirrors src/tpu_as_gpu in CodersAcademy006/tpu-as-gpu) ---

def where(condition, on_true, on_false):
    module_root = type(condition).__module__.split(".")[0]
    if module_root == "torch":
        import torch
        return torch.where(condition, on_true, on_false)
    if module_root in ("jax", "jaxlib"):
        import jax.numpy as jnp
        return jnp.where(condition, on_true, on_false)
    if module_root == "tensorflow":
        import tensorflow as tf
        return tf.where(condition, on_true, on_false)
    import numpy as np
    return np.where(condition, on_true, on_false)


def bucket_length(length, buckets):
    fitting = [b for b in sorted(buckets) if length <= b]
    if not fitting:
        raise ValueError(f"length {length} exceeds largest bucket {max(buckets)}")
    return fitting[0]


class DeferredSync:
    def __init__(self):
        self._pending = []

    def collect(self, value):
        self._pending.append(value)

    def resolve(self, sync_fn=None):
        if sync_fn is None:
            sync_fn = lambda values: [v.item() if hasattr(v, "item") else v for v in values]
        values = sync_fn(self._pending)
        self._pending = []
        return values


class StaticKVCache:
    def __init__(self, max_len, shape_per_token, zeros_fn):
        self.max_len = max_len
        self.buffer = zeros_fn((max_len, *shape_per_token))
        self.pos = 0

    def write(self, token_kv):
        if self.pos >= self.max_len:
            raise IndexError("KV cache full")
        self.buffer[self.pos] = token_kv
        self.pos += 1
        return self.buffer, self.pos


def get_torch_device():
    try:
        import torch_xla.core.xla_model as xm
        return xm.xla_device()
    except ImportError:
        pass
    import torch
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def device_kind(device):
    s = str(device)
    if "xla" in s:
        return "xla"
    if "cuda" in s:
        return "cuda"
    return "cpu"


def autocast_dtype(kind):
    return "bfloat16" if kind == "xla" else "float16"


def compile_backend(kind):
    return "openxla" if kind == "xla" else "inductor"


def seed_everything(seed):
    import random
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    try:
        import jax
        return jax.random.PRNGKey(seed)
    except ImportError:
        return None

print("tpu_as_gpu inlined library loaded")

In [ ]:
import json, time

results = {}

def record(name, fn):
    try:
        t0 = time.time()
        out = fn()
        results[name] = {"ok": True, "elapsed_s": round(time.time() - t0, 3), "detail": str(out)}
    except Exception as e:
        results[name] = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    print(name, "->", results[name])


def torch_xla_smoke():
    import torch
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    a = torch.randn(2048, 2048, device=device)
    b = torch.randn(2048, 2048, device=device)
    c = a @ b
    xm.mark_step()
    assert c.shape == (2048, 2048)
    return f"device={device} shape={tuple(c.shape)}"


def jax_smoke():
    import jax
    import jax.numpy as jnp
    assert jax.devices()[0].platform == "tpu", "no TPU backend picked up"
    key = jax.random.PRNGKey(0)
    a = jax.random.normal(key, (2048, 2048))
    b = jax.random.normal(key, (2048, 2048))
    c = jax.jit(jnp.matmul)(a, b)
    c.block_until_ready()
    return f"devices={jax.devices()} shape={c.shape}"


def tensorflow_smoke():
    import tensorflow as tf
    # TPU-VM-style runtime (no COLAB_TPU_ADDR) - resolver needs the
    # "local" TPU name. Expected to fail on a plain `pip install
    # tensorflow` - see research/prior_art.md #6.
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    with strategy.scope():
        a = tf.random.normal((2048, 2048))
        b = tf.random.normal((2048, 2048))

        @tf.function
        def matmul(a, b):
            return tf.matmul(a, b)

        c = strategy.run(matmul, args=(a, b))
    assert c.shape == (2048, 2048)
    return f"devices={tf.config.list_logical_devices('TPU')} shape={c.shape}"


def tpu_as_gpu_on_real_tpu():
    """The actual point of this run: exercise the library's logic against
    real XLA-backed tensors, not CPU/numpy stand-ins."""
    import torch
    import torch_xla.core.xla_model as xm

    device = get_torch_device()
    kind = device_kind(device)
    assert kind == "xla", f"expected xla device, got {kind}"

    cond = torch.tensor([True, False], device=device)
    out = where(cond, torch.tensor([1, 1], device=device), torch.tensor([0, 0], device=device))
    xm.mark_step()
    assert out.cpu().tolist() == [1, 0]

    assert bucket_length(5, [8, 16, 32]) == 8

    sync = DeferredSync()
    sync.collect(torch.tensor(3.0, device=device))
    sync.collect(torch.tensor(4.0, device=device))
    resolved = sync.resolve()
    xm.mark_step()
    assert resolved == [3.0, 4.0]

    cache = StaticKVCache(max_len=2, shape_per_token=(2,), zeros_fn=lambda shape: torch.zeros(shape, device=device))
    cache.write(torch.tensor([1.0, 2.0], device=device))
    xm.mark_step()

    assert autocast_dtype(kind) == "bfloat16"
    assert compile_backend(kind) == "openxla"
    seed_everything(0)

    return f"device={device} kind={kind} all tpu_as_gpu checks passed on real TPU"


record("torch_xla_smoke", torch_xla_smoke)
record("jax_smoke", jax_smoke)
record("tensorflow_smoke", tensorflow_smoke)
record("tpu_as_gpu_on_real_tpu", tpu_as_gpu_on_real_tpu)

with open("tpu_verify_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n--- summary ---")
print(json.dumps(results, indent=2))

In [ ]:
!pip install -q transformers librosa soundfile

import numpy as np
import soundfile as sf
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Synthetic tone - a stand-in for a real clip, just to prove the pipeline
# runs end to end. Produces near-silence-like output, which is exactly
# what hits the degenerate-sequence edge case below.
sr = 16000
t = np.linspace(0, 2, 2 * sr, endpoint=False)
tone = 0.1 * np.sin(2 * np.pi * 440 * t).astype("float32")
sf.write("sample.wav", tone, sr)

device = get_torch_device()
processor = WhisperProcessor.from_pretrained("openai/whisper-base")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").to(device)

audio, sr = librosa.load("sample.wav", sr=16000)
inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
input_features = inputs.input_features.to(device)

import torch_xla.core.xla_model as xm

def whisper_smoke():
    t0 = time.time()
    # min_new_tokens guards a real eager-vs-XLA divergence: short/silent
    # audio can decode to a 1-token sequence, and transformers' own
    # segment-retrieval code does a negative-index slice on it that eager
    # PyTorch clamps silently but torch_xla's compiled slicing throws on.
    # See research/prior_art.md #7.
    generated_ids = model.generate(input_features, min_new_tokens=5)
    xm.mark_step()
    elapsed = time.time() - t0
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return f"device={device} elapsed={elapsed:.2f}s text={text!r}"

record("whisper_tpu", whisper_smoke)
print(json.dumps(results, indent=2))


In [ ]:
import torch_xla.distributed.parallel_loader as pl
from torch.utils.data import DataLoader, TensorDataset


def get_memory_info(device_kind_, dev=None):
    if device_kind_ == "xla":
        return dict(xm.get_memory_info(dev))
    if device_kind_ == "cuda":
        free, total = torch.cuda.mem_get_info(dev)
        return {"bytes_used": total - free, "bytes_limit": total}
    return None


def clear_cache(device_kind_):
    if device_kind_ == "cuda":
        torch.cuda.empty_cache()
        return True
    return False


def profiling_smoke():
    info = get_memory_info("xla", device)
    assert info["bytes_used"] > 0
    assert clear_cache("xla") is False  # honestly a no-op on XLA
    return f"memory_info={info}"


def mp_device_loader_smoke():
    ds = TensorDataset(torch.randn(16, 4))
    loader = DataLoader(ds, batch_size=4)
    mp_loader = pl.MpDeviceLoader(loader, device)
    n = 0
    for (batch,) in mp_loader:
        assert device_kind(batch.device) == "xla"
        n += 1
    xm.mark_step()
    return f"{n} batches on {device_kind(batch.device)}"


record("profiling_memory_info", profiling_smoke)
record("mp_device_loader", mp_device_loader_smoke)
print(json.dumps(results, indent=2))


In [ ]:
try:
    from google.colab import files
    files.download("tpu_verify_results.json")
except ImportError:
    print("not running in Colab, results saved at ./tpu_verify_results.json")